In [ ]:
import torch
import pandas as pd
import numpy as np
import os
from load_data_function import generate_UCSD_Nissan_dataset, get_random_battery_cycle_and_soh_same_distribution, downsample_all_battery_cycle, fig_plot,save_data, load_data, battery_soh_plot, smooth_soh
import pickle
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from d2l import torch as d2l
import sys
sys.path.append("..")  # 向上级目录查找
from functions.config import Config

观察得到，T取CALCE或IECON数据集


In [ ]:
def generate_transfer_T_dataset(dataset_path, new_length, config, package,dataset,max_cycle=None,target_dataset=False):

    cell_cycles=[]
    train_cell_cycles=[]
    test_cell_cycles=[]
    T_test_data_dict={}
    T_test_soh_dict={}
    pkl_path = f'{dataset_path}/downsampled_data_{new_length}/{package}/{new_length}_{dataset}_all_battery_id_data.pkl'
    soh_path = f'{dataset_path}\\downsampled_data_{new_length}\\{package}\\{new_length}_{dataset}_all_battery_id_soh.pkl'
    if 1==1:
        with open(soh_path, 'rb') as file:
            soh_data = pickle.load(file)
            keys = list(soh_data.keys())
            all_battery_ids = np.arange(len(keys)) + 1
            #print(all_battery_ids)
            soh=[]
            train_soh=[]
            test_soh=[]
        with open(pkl_path, 'rb') as file:
            data = pickle.load(file)
            new_data = []
            train_data=[]
            test_data=[]

            for i in all_battery_ids:
                num=0
                i_battery_data_soh = np.array(soh_data[f'battery_{i}']).reshape(-1, 1)
                for j in range(len(i_battery_data_soh)):
                    if np.average(i_battery_data_soh[j:j+25])<=0.8:
                        num=j
                        break
                if num!=0:
                    i_battery_data_soh=i_battery_data_soh[:num]
                    soh.append(i_battery_data_soh)
                    i_battery_data = np.array(data[f'battery_{i}'])
                    i_battery_data = i_battery_data[:num]
                    new_data.append(i_battery_data)
                    cell_cycles.append(len(i_battery_data_soh))
                else:
                    soh.append(i_battery_data_soh)
                    i_battery_data = np.array(data[f'battery_{i}'])
                    new_data.append(i_battery_data)
                    cell_cycles.append(len(i_battery_data_soh))
                if target_dataset:

                    if i in config.train_battery_id:
                        train_cell_cycles.append(len(i_battery_data_soh))
                        train_data.append(i_battery_data)
                        train_soh.append(i_battery_data_soh)
                    else:
                        test_cell_cycles.append(len(i_battery_data_soh))
                        test_data.append(i_battery_data)
                        test_soh.append(i_battery_data_soh)
                        T_test_data_dict[f'battery_{i-3}'] = i_battery_data
                        T_test_soh_dict[f'battery_{i-3}'] = i_battery_data_soh

                if max_cycle is not None:
                    if len(np.concatenate(new_data, axis=0))>max_cycle:
                        break
            soh = np.concatenate(soh, axis=0)
            new_data = np.concatenate(new_data, axis=0)
            if target_dataset:
                train_data = np.concatenate(train_data, axis=0)
                test_data = np.concatenate(test_data, axis=0)
                train_soh = np.concatenate(train_soh, axis=0)
                test_soh = np.concatenate(test_soh, axis=0)

    new_data = new_data.astype(np.float32)
    soh = soh.astype(np.float32)
    if target_dataset:
        train_soh = train_soh.astype(np.float32)
        test_soh = test_soh.astype(np.float32)
        train_data = train_data.astype(np.float32)
        test_data = test_data.astype(np.float32)

    return cell_cycles,train_cell_cycles,test_cell_cycles,new_data,train_data,test_data,soh,train_soh,test_soh, T_test_data_dict,T_test_soh_dict


In [ ]:
transformed_data_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data'
final_data_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\T_Toyota_S10_T1'
config = Config()
new_length=256
#target_dataset = 'SNL_NMC'  # 选择目标数据集
#target_dataset= 'TongJi'  # 选择目标数据集
#target_dataset='Oxford'
#target_dataset='HNEL'
#target_dataset='NASA'
target_dataset='Toyota_MIT'
S_data_dict={}
S_soh_dict={}
dict_S_cell_cycles={}
T_cell_cycles=[]
T_train_cell_cycles=[]
T_test_cell_cycles=[]
max_cycle=2000    #每个电池包中电池循环数的限制
#max_cycle=None

for i,dataset in enumerate(os.listdir(transformed_data_path)):
    print(f'dataset: {dataset}\t,{config.All_Dataset[i]}')
    #S_data_dict[config.All_Dataset[i]]=[]
    #S_soh_dict[config.All_Dataset[i]]=[]
    if config.All_Dataset[i] == target_dataset:
        for j,package in enumerate(os.listdir(os.path.join(transformed_data_path, dataset, f'downsampled_data_{new_length}'))):
            if j==1:
                cell_cycles,train_cell_cycles,test_cell_cycles,data,train_data,test_data,soh,train_soh,test_soh,T_test_data_dict,T_test_soh_dict = generate_transfer_T_dataset(os.path.join(transformed_data_path, dataset), new_length, config, package,config.All_Dataset[i],8000,target_dataset=True)
                T_data = data
                T_soh = soh
                T_cell_cycles=cell_cycles

                T_train_data = train_data
                T_train_soh = train_soh
                T_train_cell_cycles=train_cell_cycles

                T_test_data = test_data
                T_test_soh = test_soh
                T_test_cell_cycles=test_cell_cycles
            else:
                continue
                cell_cycles,train_cell_cycles,test_cell_cycles,data,train_data,test_data,soh,train_soh,test_soh,T_test_data_dict,T_test_soh_dict = generate_transfer_T_dataset(os.path.join(transformed_data_path, dataset), new_length, config, package,config.All_Dataset[i],max_cycle,target_dataset=True)
                T_data = np.concatenate((T_data, data), axis=0)
                T_soh = np.concatenate((T_soh, soh), axis=0)
                T_cell_cycles.extend(cell_cycles)

                #T_train_data = np.concatenate((T_train_data, train_data), axis=0)
                #T_train_soh = np.concatenate((T_train_soh, train_soh), axis=0)
                #T_train_cell_cycles.extend(train_cell_cycles)

                #T_test_data = np.concatenate((T_test_data, test_data), axis=0)
                #T_test_soh = np.concatenate((T_test_soh, test_soh), axis=0)
                #T_test_cell_cycles.extend(test_cell_cycles)
                T_test_data = test_data
                T_test_soh = test_soh
                T_test_cell_cycles=test_cell_cycles

        T_soh = T_soh.reshape(-1, 1)
        print(f'T_data_shape: {T_data.shape}, T_soh_shape: {T_soh.shape}')
        print(f'T_train_data_shape: {T_train_data.shape}, T_train_soh_shape: {T_train_soh.shape}')
        print(f'T_test_data_shape: {T_test_data.shape}, T_test_soh_shape: {T_test_soh.shape}')
        print(f'T_test_data_dict_keys: {T_test_data_dict.keys()}')
        #print(f'T_test_data_shape: {T_test_data_dict["battery_5"].shape}')
    else:
        S_data_dict[config.All_Dataset[i]]=[]
        S_soh_dict[config.All_Dataset[i]]=[]
        dict_S_cell_cycles[config.All_Dataset[i]]=[]
        for j,package in enumerate(os.listdir(os.path.join(transformed_data_path, dataset, f'downsampled_data_{new_length}'))):
            cell_cycles,_,_,data,_,_,soh,_,_,_,_ = generate_transfer_T_dataset(os.path.join(transformed_data_path, dataset), new_length, config, package,config.All_Dataset[i],max_cycle)
            if j==0:
                S_data = data
                S_soh = soh
                dict_S_cell_cycles[config.All_Dataset[i]]=cell_cycles
            elif j==3:
                break

            else:

                S_data = np.concatenate((S_data, data), axis=0)
                S_soh = np.concatenate((S_soh, soh), axis=0)
                dict_S_cell_cycles[config.All_Dataset[i]].extend(cell_cycles)

        S_soh = S_soh.reshape(-1, 1)
        S_data_dict[config.All_Dataset[i]]=S_data
        S_soh_dict[config.All_Dataset[i]]=S_soh
        print(f'S_data_shape: {S_data.shape}, S_soh_shape: {S_soh.shape}')

In [ ]:
small_data=False
max_length=100
if small_data==True:
    for S_domian in S_data_dict:
        if len(S_data_dict[S_domian]) > max_length:
            print(f'{S_domian} 数据集数据长度大于{max_length}, S_data长度: {len(S_data_dict[S_domian])}, S_soh长度: {len(S_soh_dict[S_domian])}')
            S_data_dict[S_domian] = S_data_dict[S_domian][:max_length]
            S_soh_dict[S_domian] = S_soh_dict[S_domian][:max_length]

if small_data==True:
    T_data=T_data[:max_length]
    T_soh=T_soh[:max_length]


In [ ]:
for dataset in S_data_dict.keys():
    if len(S_data_dict[dataset])!= len(S_soh_dict[dataset]):
        print(f'{dataset} 数据集数据长度不一致, S_data长度: {len(S_data_dict[dataset])}, S_soh长度: {len(S_soh_dict[dataset])}')


In [ ]:
#final_data_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\T_XJTU_S_HNEL_IECON_Toyota_XJTU_data_S3_T1_2'
os.makedirs(final_data_path,exist_ok=True)
save_data(S_data_dict,os.path.join(final_data_path,'S_data_dict.pkl'))
save_data(S_soh_dict,os.path.join(final_data_path,'S_soh_dict.pkl'))
save_data(T_data,os.path.join(final_data_path,'T_data.pkl'))
save_data(T_train_data,os.path.join(final_data_path,'T_train_data.pkl'))
save_data(T_test_data,os.path.join(final_data_path,'T_test_data.pkl'))
save_data(T_soh,os.path.join(final_data_path,'T_soh.pkl'))
save_data(T_train_soh,os.path.join(final_data_path,'T_train_soh.pkl'))
save_data(T_test_soh,os.path.join(final_data_path,'T_test_soh.pkl'))
save_data(dict_S_cell_cycles,os.path.join(final_data_path,'dict_S_cell_cycles.pkl'))
save_data(T_cell_cycles,os.path.join(final_data_path,'T_cell_cycles.pkl'))
save_data(T_train_cell_cycles,os.path.join(final_data_path,'T_train_cell_cycles.pkl'))
save_data(T_test_cell_cycles,os.path.join(final_data_path,'T_test_cell_cycles.pkl'))
save_data(T_test_data_dict,os.path.join(final_data_path,'T_test_data_dict.pkl'))
save_data(T_test_soh_dict,os.path.join(final_data_path,'T_test_soh_dict.pkl'))

In [ ]:
S_data_dict=load_data(os.path.join(final_data_path,'S_data_dict.pkl'))
S_soh_dict=load_data(os.path.join(final_data_path,'S_soh_dict.pkl'))
T_data=load_data(os.path.join(final_data_path,'T_data.pkl'))
T_data_soh=load_data(os.path.join(final_data_path,'T_soh.pkl'))
print(f'S_data_dict: {S_data_dict.keys()}, T_data_shape: {T_data.shape}')
def generate_dataset(S_data_dict, S_soh_dict, T_data, T_data_soh, config):
    dict_S_loader={}
    for domian in S_data_dict.keys():
        S_data=S_data_dict[domian]
        S_data_soh=S_soh_dict[domian]
        S_x = torch.from_numpy(S_data)
        S_y = torch.from_numpy(S_data_soh)
        S_loader = DataLoader(TensorDataset(S_x, S_y), batch_size=config.Batch_size, shuffle=True,
                              drop_last=False)
        dict_S_loader[domian]=S_loader
    T_x = torch.from_numpy(T_data)
    T_y = torch.from_numpy(T_data_soh)


    T_loader = DataLoader(TensorDataset(T_x, T_y), batch_size=config.Batch_size, shuffle=False,
                              drop_last=False)
    return dict_S_loader,T_loader

In [ ]:
print(T_data.shape)
print(T_train_data.shape)
